# Mode-Specific Transport Effects on Sectoral Development

**Purpose.** Estimate and visualize associations between transport modes and agricultural, industrial, and service-sector outcomes.

**Inputs.** Transport, sectoral value-added, macroeconomic, and boundary data.

**Outputs.** Mode-specific sectoral-effect estimates and figures.

Run the code cells from top to bottom after updating the local paths described in the repository README. The notebook has no stored outputs; figures and tables are generated when it is executed.

In [ ]:
# Cell | Dependencies and visual-design settings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle

import geopandas as gpd
import contextily as ctx

from scipy.stats import pearsonr
from scipy import stats  

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 25
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['figure.dpi'] = 300

region_palette = {
    'Central Africa': '#F4831F',
    'East Africa':    '#9B8EC4',
    'North Africa':   '#4DBBD5',
    'Southern Africa':'#5cbe7d',
    'West Africa':    '#E64B35',
}

sector_palette  = {'Agriculture': '#27AE60', 'Industry': '#2980B9', 'Services': '#C0392B'}

sector_markers  = {'Agriculture': 'o', 'Industry': 's', 'Services': '^'}



In [ ]:
# Cell | Regional classification and transport-data preparation
# Map countries to the five African subregions for stratified analysis.
region_mapping = {
    'North Africa': ['Algeria', 'Egypt', 'Libya', 'Mauritania', 'Morocco', 'Tunisia'],
    'West Africa': ['Benin', 'Burkina Faso', "Ivory Coast", 'Gambia', 'Ghana',
                    'Guinea', 'Guinea-Bissau', 'Liberia', 'Mali', 'Niger',
                    'Nigeria', 'Senegal', 'Sierra Leone', 'Togo'],
    'Central Africa': ['Cameroon', 'Central African Republic', 'Chad', 'Republic of the Congo',
                       'Democratic Republic of the Congo', 'Equatorial Guinea', 'Gabon'],
    'East Africa': ['Burundi', 'Djibouti', 'Eritrea', 'Ethiopia', 'Kenya',
                    'Rwanda', 'Somalia', 'South Sudan', 'Sudan', 'United Republic of Tanzania', 'Uganda'],
    'Southern Africa': ['Angola', 'Botswana', 'eSwatini', 'Lesotho', 'Madagascar', 'Malawi',
                        'Mozambique', 'Namibia', 'South Africa', 'Zambia', 'Zimbabwe']
}

# Attach the regional classification to each country.
country_to_region = {c: r for r, cs in region_mapping.items() for c in cs}

all_mapped_countries = [c for cs in region_mapping.values() for c in cs]

df_transport_abs = pd.read_excel(
    r'E:\Africa_Infrastructure_Data\transport_statistics_2015_2025.xlsx'
)

df_transport_abs = df_transport_abs[
    (df_transport_abs['year'] >= 2015) & (df_transport_abs['year'] <= 2024)
].copy()

# Attach the regional classification to each country.
df_transport_abs['region'] = df_transport_abs['country'].map(country_to_region)

# Work on a copy to preserve the source table.
df_transport_abs = df_transport_abs[df_transport_abs['country'].isin(all_mapped_countries)].copy()

road_type_cols = ['road_motorway_km', 'road_trunk_km', 'road_primary_km',
                  'road_secondary_km', 'road_tertiary_km']

df_transport_abs['total_road_km'] = df_transport_abs[road_type_cols].sum(axis=1)



In [ ]:
# Cell | World Bank sectoral value-added data
# Specify World Bank value-added series for agriculture, industry and services.
industry_files = {
    'Agriculture': r'E:\Africa_Infrastructure_Data\world_bank\API_NV.AGR.TOTL.CD_DS2_en_csv_v2_723\API_NV.AGR.TOTL.CD_DS2_en_csv_v2_723.csv',
    'Industry':    r'E:\Africa_Infrastructure_Data\world_bank\API_NV.IND.TOTL.KD_DS2_en_csv_v2_14108\API_NV.IND.TOTL.KD_DS2_en_csv_v2_14108.csv',
    'Services':    r'E:\Africa_Infrastructure_Data\world_bank\API_NV.SRV.TOTL.KD_DS2_en_csv_v2_14118\API_NV.SRV.TOTL.KD_DS2_en_csv_v2_14118.csv'
}

wb_to_transport = {
    'Congo, Dem. Rep.': 'Democratic Republic of the Congo',
    'Congo, Rep.': 'Republic of the Congo',
    'Tanzania': 'United Republic of Tanzania',
    'Eswatini': 'eSwatini',
    "Cote d'Ivoire": 'Ivory Coast',
    'Egypt, Arab Rep.': 'Egypt',
    'Gambia, The': 'Gambia'
}

year_cols = [str(y) for y in range(2015, 2025)]

industry_data_list = []
valid_countries_per_sector = {}  # Define the lookup values used in the analysis.

for sector, fpath in industry_files.items():
    df_s = pd.read_csv(fpath, skiprows=4)
    
    df_s['Country Name'] = df_s['Country Name'].replace(wb_to_transport)
    
    df_s = df_s[df_s['Country Name'].isin(all_mapped_countries)][
        ['Country Name', 'Country Code', 'Indicator Name'] + year_cols
    ]
    
    df_long = df_s.melt(id_vars=['Country Name', 'Country Code', 'Indicator Name'],
                        value_vars=year_cols, var_name='year', value_name='value')
    
    df_long['year'] = df_long['year'].astype(int)
    
    df_long['sector'] = sector

    valid = [c for c in df_s['Country Name'].unique()
             if df_long[(df_long['Country Name']==c) & df_long['value'].notna()].shape[0] > 3]
    
    valid_countries_per_sector[sector] = valid
    
    industry_data_list.append(df_long[df_long['Country Name'].isin(valid)])

# Stack the equation-specific observations for joint estimation.
df_industry = pd.concat(industry_data_list, ignore_index=True)

# Attach the regional classification to each country.
df_industry['region'] = df_industry['Country Name'].map(country_to_region)

In [ ]:
# Cell | AfDB macroeconomic data and country-name harmonization
# ==========================================
# Read AfDB macroeconomic indicators and harmonize country names.
df_economic = pd.read_csv(r'E:\Africa_Infrastructure_Data\afdb\ObservationData_fhkvkjf_constant.csv')

afdb_map = {
    "Congo, Dem. Rep.": "Democratic Republic of the Congo",
    "Congo, Rep.": "Republic of the Congo",
    "Tanzania": "United Republic of Tanzania",
    "Eswatini": "eSwatini",
    "Cote d'Ivoire": "Ivory Coast"
}

df_economic['country'] = df_economic['country'].replace(afdb_map)

# Work on a copy to preserve the source table.
df_population = df_economic[df_economic['indicator'] == 'Population, total'].copy()

df_population = df_population[(df_population['Date'] >= 2015) & (df_population['Date'] <= 2024)]

df_population = df_population[['country', 'Date', 'Value']].rename(
    columns={'Date': 'year', 'Value': 'Population'}
)

# ==========================================
# Reshape the long table into indicator columns.
df_industry_pivot = df_industry.pivot_table(
    index=['Country Name', 'year', 'region'], columns='sector', values='value', aggfunc='first'
).reset_index()

df_industry_pivot.columns.name = None

df_industry_pivot = df_industry_pivot.rename(columns={
    'Agriculture': 'Agriculture_value_added',
    'Industry':    'Industry_value_added',
    'Services':    'Services_value_added'
})

# ==========================================
# Merge records using the shared country-year keys.
df_merged = pd.merge(df_transport_abs, df_industry_pivot,
                     left_on=['country', 'year'], right_on=['Country Name', 'year'], how='inner')


df_merged = pd.merge(df_merged, df_population, on=['country', 'year'], how='left')

df_merged['region'] = df_merged['region_x']

df_merged = df_merged.drop(columns=['region_x', 'region_y'], errors='ignore')



In [ ]:
# Cell | Country-year analytical panel and growth correlations
transport_6cols   = ['total_road_km', 'motorway_trunk_paved_km', 'railway_length_km',
                     'railway_station_area_km2', 'port_area_km2', 'airport_area_km2']
transport_6labels = ['Road', 'Paved Highway', 'Railway', 'Railway Station', 'Port', 'Aeroway']

industry_cols3    = ['Agriculture_value_added', 'Industry_value_added', 'Services_value_added']
industry_labels3  = ['Agriculture', 'Industry', 'Services']

transport_gr_cols = [f'{c}_gr' for c in transport_6cols]
industry_gr_cols  = [f'{c}_gr' for c in industry_cols3]

# Calculate percentage growth relative to the selected baseline year.
def calc_growth_rate(val, base_val):
    if pd.isna(base_val) or pd.isna(val):
        return np.nan
    if base_val == 0:
        return np.nan 
    return (val - base_val) / abs(base_val) * 100

growth_records = []

for country in df_merged['country'].unique():
    df_c = df_merged[df_merged['country'] == country].sort_values('year')
    
    base = df_c[df_c['year'] == 2015]
    if base.empty:
        continue
        
    base_row = base.iloc[0]
    
    for _, row in df_c.iterrows():
        rec = {'country': country, 'year': row['year']}  # Define helper functions.
        
        # Estimate the model for each transport indicator.
        for col in transport_6cols + industry_cols3:
                rec[f'{col}_gr'] = calc_growth_rate(row[col], base_row[col])
        growth_records.append(rec)

df_growth_country = pd.DataFrame(growth_records)


corr_gr_records = []

for country in df_growth_country['country'].unique():
    df_c = df_growth_country[(df_growth_country['country'] == country) & (df_growth_country['year'] >= 2016)].sort_values('year')
    
    # Estimate the model for each transport indicator.
    for tc_gr, tc_orig in zip(transport_gr_cols, transport_6cols):
        for ic_gr, il in zip(industry_gr_cols, industry_labels3):
            
            # Retain complete observations required for estimation.
            sub = df_c[[tc_gr, ic_gr]].dropna()
            
            if len(sub) >= 4 and sub[tc_gr].std() > 0 and sub[ic_gr].std() > 0:
                r_val, p_val = pearsonr(sub[tc_gr], sub[ic_gr])
            else:
                r_val, p_val = np.nan, np.nan
                
            corr_gr_records.append({'country': country, 'transport': tc_orig,
                                    'industry': il, 'r': r_val, 'p': p_val})

# Calculate country-year growth correlations between transport modes and sectoral outcomes.
df_corr_gr = pd.DataFrame(corr_gr_records)


In [ ]:
# Cell | Spatial reference data and map coordinates

# Read the spatial boundary layer.
gdf = gpd.read_file(r'E:\Africa_Infrastructure_Data\boundaries_project_49\Country_Subregion.shp')

gdf['cx'] = gdf.geometry.centroid.x
gdf['cy'] = gdf.geometry.centroid.y

xmin_g, ymin_g, xmax_g, ymax_g = gdf.total_bounds

map_w = xmax_g - xmin_g
map_h = ymax_g - ymin_g

centroid_map = gdf.set_index('国家')[['cx', 'cy']]

# Calculate country-year growth correlations between transport modes and sectoral outcomes.
df_corr_gr['cx'] = df_corr_gr['country'].map(centroid_map['cx'])
df_corr_gr['cy'] = df_corr_gr['country'].map(centroid_map['cy'])


MANUAL_OFFSETS = {
    'Gambia':            (-0.08,  0.05,  0.35),
    'Guinea-Bissau':     (-0.07, -0.01, -0.30),
    'Sierra Leone':      (-0.07, -0.04,  0.30),
    'Liberia':           (-0.06, -0.07, -0.25),
    'Togo':              ( 0.03, -0.07, -0.25),
    'Equatorial Guinea': (-0.05, -0.03,  0.30),
    'Burundi':           ( 0.20, -0.05, -0.35),
    'Mali':              ( 0.00,  0.06,  0.0),
    'Cameroon':          ( 0.01,  0.04,  0.0),
    'Ghana':             ( 0.00, -0.02,  0.0),
    'South Africa':      (-0.04, -0.04,  0.0),
    'Uganda':            ( 0.015,  0.025,  0.0),
    'Tunisia':           ( 0.00, -0.02,  0.0),
}

NO_ARROW_COUNTRIES = {'Mali', 'Cameroon', 'Ghana', 'South Africa', 'Uganda', 'Tunisia'}

In [ ]:
# Cell | Correlation-result inspection
df_corr_gr

In [ ]:
# Cell | Correlation-dependent colour scaling
# Scale map colours by the absolute correlation magnitude.
def r_to_color(base_hex, r_abs):
    """Adjust a base colour by correlation strength."""
    h, s, v = mcolors.rgb_to_hsv(mcolors.to_rgb(base_hex))
    
    if r_abs >= 0.6:
        t = (r_abs - 0.6) / 0.4
        s_out = 0.70 + t * 0.30
        v_out = 0.75 - t * 0.20
    else:
        t = r_abs / 0.6
        s_out = 0.10 + t * 0.60
        v_out = 0.95 - t * 0.10
        
    return mcolors.hsv_to_rgb([h, s_out, v_out])

def add_compass(ax, x=0.08, y=0.93, size=0.038):
    """Helper function."""
    from matplotlib.patches import Polygon as P  # Import the dependency used in this section.
    
    ax.add_patch(P(np.array([[x, y+size], [x-size*.4, y], [x+size*.4, y]]),
                   closed=True, fc='#222', ec='white', lw=.5,
                   transform=ax.transAxes, zorder=12))
                   
    ax.add_patch(P(np.array([[x, y-size], [x-size*.4, y], [x+size*.4, y]]),
                   closed=True, fc='white', ec='#222', lw=.5,
                   transform=ax.transAxes, zorder=12))
                   
    ax.text(x, y+size*1.3, 'N', transform=ax.transAxes,
            ha='center', va='center', fontsize=20, fontweight='bold',
            fontfamily='Times New Roman', color='#222', zorder=13)

def add_scalebar(ax, seg_km=1000, n_seg=2, map_width_m=None):
    """Helper function."""
    bw = (seg_km * n_seg * 1000) / map_width_m if map_width_m else 0.30
    bx, by, bh = 0.04, 0.07, 0.012
    sw = bw / n_seg
    
    for i in range(n_seg):
        ax.add_patch(Rectangle((bx+i*sw, by), sw, bh,
                               fc='#111' if i%2==0 else '#eee', ec='#333', lw=.6,
                               transform=ax.transAxes, zorder=11, clip_on=False))
                               
    for i, km in enumerate([0, seg_km, seg_km*n_seg]):
        ax.text(bx+i*sw, by-.008, f'{km}' if km < seg_km*n_seg else f'{km} km',
                transform=ax.transAxes, ha='center', va='top', fontsize=20,
                fontfamily='Times New Roman', color='#333', zorder=12)

def draw_bars(ax, pcx, pcy, df_c, industry_labels3, sector_palette,
              BAR_W_ONE, BAR_MAX_H, TOTAL_BAR_W):
    """Ignore relation features."""
    
    ax.plot([pcx - TOTAL_BAR_W*.65, pcx + TOTAL_BAR_W*.65], [pcy, pcy],
            color='#222', linewidth=1.4, zorder=6, solid_capstyle='round')
            
    for j, il in enumerate(industry_labels3):
        df_il = df_c[df_c['industry'] == il]
        if df_il.empty:
            continue
            
        r, p = df_il['r'].iloc[0], df_il['p'].iloc[0]
        bar_x = pcx - TOTAL_BAR_W/2 + j*BAR_W_ONE
        
        if np.isnan(r):
            ax.add_patch(Rectangle((bar_x, pcy), BAR_W_ONE, BAR_MAX_H*.05,
                                   fc='#ddd', ec='none', alpha=.5, zorder=4))
            continue
            
        bar_h = abs(r) * BAR_MAX_H
        bar_y = pcy if r >= 0 else pcy - bar_h
        
        ax.add_patch(Rectangle((bar_x, bar_y), BAR_W_ONE, bar_h,
                                fc=r_to_color(sector_palette[il], abs(r)),  # Define helper functions.
                                ec='white', lw=0.1,
                                hatch=None if r >= 0 else '///', 
                                alpha=0.95, zorder=5))
                                
        if not np.isnan(p) and p < 0.05:
            offset = BAR_MAX_H * 0.01
            sy = bar_y + bar_h + offset if r >= 0 else bar_y - offset
            ax.text(bar_x + BAR_W_ONE/2, sy, '*',
                    ha='center', va='bottom' if r >= 0 else 'top',
                    fontsize=11, fontfamily='Times New Roman',
                    color='#111', fontweight='bold', zorder=8)



In [ ]:
# Cell | Map layout utilities
import matplotlib.transforms as mtransforms

pad_x   = map_w * 0.04
pad_top = map_h * 0.03
pad_bot = map_h * (-0.06)
display_w = map_w + 2*pad_x
display_h = map_h + pad_top + pad_bot
data_aspect = display_w / display_h  # Configure parameters.

FIG_W    = 24
MAP_LEFT, MAP_RIGHT = 0.01, 0.99
col_w_frac  = (MAP_RIGHT - MAP_LEFT - 2*0.012) / 3
row_h_inch  = FIG_W * col_w_frac / data_aspect

row3_h_inch = 4.5
gap_row3    = 0.50
legend_inch = 2.10

FIG_H = 2*row_h_inch + gap_row3 + row3_h_inch + legend_inch

map_top_frac  = 1.0
map_bot_frac  = (row3_h_inch + gap_row3 + legend_inch) / FIG_H
row3_top_frac = (row3_h_inch + legend_inch) / FIG_H
row3_bot_frac = legend_inch / FIG_H

VX0, VX1 = xmin_g - pad_x, xmax_g + pad_x
VY0, VY1 = ymin_g - pad_bot, ymax_g + pad_top

BAR_W_ONE   = map_w * 0.013
BAR_MAX_H   = map_h * 0.06
TOTAL_BAR_W = BAR_W_ONE * len(industry_labels3)
countries_with_data = set(df_merged['country'].unique())

from scipy.stats import norm as sp_norm

def _calc_growth_rate(val, base_val):
    """Helper function."""
    if pd.isna(base_val) or pd.isna(val): return np.nan
    if base_val == 0: return val - base_val
    return (val - base_val) / abs(base_val) * 100

df_regional_ts_pre = df_merged.groupby(['region', 'year']).agg(
    {**{c: 'mean' for c in transport_6cols}, **{c: 'mean' for c in industry_cols3}}
).reset_index()

reg_ts_gr_pre = []
for region in df_regional_ts_pre['region'].unique():
    df_r = df_regional_ts_pre[df_regional_ts_pre['region'] == region].sort_values('year')
    base = df_r[df_r['year'] == 2015]
    if base.empty: continue
    base_row = base.iloc[0]
    for _, row in df_r.iterrows():
        rec = {'region': region, 'year': row['year']}
        for col in transport_6cols + industry_cols3:
            rec[f'{col}_gr'] = _calc_growth_rate(row[col], base_row[col])
        reg_ts_gr_pre.append(rec)
df_growth_reg_pre = pd.DataFrame(reg_ts_gr_pre)

regions_order_pre = sorted(df_merged['region'].unique())
region_corr_pre = {}
for region in regions_order_pre:
    df_r = df_growth_reg_pre[(df_growth_reg_pre['region'] == region) & (df_growth_reg_pre['year'] >= 2016)]
    records = []
    for tc_gr, tl in zip(transport_gr_cols, transport_6labels):
        for ic_gr, il in zip(industry_gr_cols, industry_labels3):
            sub = df_r[[tc_gr, ic_gr]].dropna()
            if len(sub) >= 4 and sub[tc_gr].std() > 0 and sub[ic_gr].std() > 0:
                r_val, p_val = pearsonr(sub[tc_gr], sub[ic_gr])
            else:
                r_val, p_val = np.nan, np.nan
            records.append({'transport': tl, 'industry': il, 'r': r_val, 'p': p_val})
    region_corr_pre[region] = pd.DataFrame(records)

region_markers = {
    'Central Africa': 'o', 'East Africa': 's',
    'North Africa': '^', 'Southern Africa': 'D', 'West Africa': 'P',
}

roman_labels = ['I', 'II', 'III', 'IV', 'V', 'VI']

fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor='white')

gs_map = gridspec.GridSpec(2, 3, figure=fig,
    left=MAP_LEFT, right=MAP_RIGHT,
    top=map_top_frac, bottom=map_bot_frac,
    hspace=0.015, wspace=0.001)

gs_row3 = gridspec.GridSpec(1, 3, figure=fig,
    left=0.06, right=0.98,
    top=row3_top_frac, bottom=row3_bot_frac,
    wspace=0.27)

map_axes  = [fig.add_subplot(gs_map[r, c]) for r in range(2) for c in range(3)]
row3_axes = [fig.add_subplot(gs_row3[0, i]) for i in range(3)]

# Estimate the model for each transport indicator.
for map_idx, (tc, tl) in enumerate(zip(transport_6cols, transport_6labels)):
    ax = map_axes[map_idx]
    ax.set_xlim(VX0, VX1)
    ax.set_ylim(VY0, VY1)

    gdf['plot_color'] = gdf['国家'].apply(
        lambda c: region_palette.get(
            gdf.loc[gdf['国家'] == c, '分区'].values[0], '#ccc')
        if c in countries_with_data else 'white')
        
    gdf.plot(ax=ax, color=gdf['plot_color'], edgecolor='none', alpha=0.45, zorder=2)
    
    # Continue gracefully if this estimation step fails.
    try:
        ctx.add_basemap(ax, crs=gdf.crs.to_string(),
                        source=ctx.providers.CartoDB.PositronNoLabels,
                        zoom='auto', zorder=1, attribution=False)
    except Exception:
        ax.set_facecolor('#F5F5F0')
        
    gdf.boundary.plot(ax=ax, color='#444', linewidth=0.45, zorder=3)

    # Calculate country-year growth correlations between transport modes and sectoral outcomes.
    df_sub = df_corr_gr[(df_corr_gr['transport'] == tc) &
                        df_corr_gr['cx'].notna() & df_corr_gr['cy'].notna()]

    for country in df_sub['country'].unique():
        df_c = df_sub[df_sub['country'] == country]
        cx, cy = df_c['cx'].iloc[0], df_c['cy'].iloc[0]
        if np.isnan(cx) or np.isnan(cy):
            continue
            
        if country in MANUAL_OFFSETS:
            dx_frac, dy_frac, arrow_rad = MANUAL_OFFSETS[country]
            pcx = np.clip(cx + dx_frac*map_w, VX0 + TOTAL_BAR_W*.65, VX1 - TOTAL_BAR_W*.65)
            pcy = np.clip(cy + dy_frac*map_h, VY0 + BAR_MAX_H, VY1 - BAR_MAX_H)
            
            draw_bars(ax, pcx, pcy, df_c, industry_labels3, sector_palette,
                      BAR_W_ONE, BAR_MAX_H, TOTAL_BAR_W)
                      
            # Skip the iteration when a required variable is unavailable.
            if country not in NO_ARROW_COUNTRIES:
                ax.annotate('', xy=(cx, cy), xytext=(pcx, pcy),
                            arrowprops=dict(arrowstyle='->', color='#333333', lw=1.8,
                                           connectionstyle=f'arc3,rad={arrow_rad}',
                                           mutation_scale=14),
                            zorder=9)
        else:
            draw_bars(ax, cx, cy, df_c, industry_labels3, sector_palette,
                      BAR_W_ONE, BAR_MAX_H, TOTAL_BAR_W)

    ax.axis('off')
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_color('#999'); sp.set_linewidth(0.8)
    ax.set_frame_on(True)
    
    ax.text(0.015, 0.985, chr(ord('a') + map_idx), transform=ax.transAxes,
            fontsize=35, fontweight='bold', va='top',
            fontfamily='Arial', color='#111', zorder=15)
    ax.text(0.985, 0.985, tl, transform=ax.transAxes,
            ha='right', va='top', fontsize=27.5, fontweight='bold',
            fontfamily='Arial', color='#111', zorder=15)
            
    add_compass(ax)
    add_scalebar(ax, map_width_m=(VX1 - VX0))

x_pos     = np.arange(len(transport_6labels))
n_regions = len(regions_order_pre)
x_jitter  = np.linspace(-0.28, 0.28, n_regions)
FSIZE     = 20

ghi_title = ['AVA','IVA','SVA']
for col_idx, industry in enumerate(industry_labels3):
    ax = row3_axes[col_idx]
    ax.set_facecolor('#FAFAFA')
    for sp in ax.spines.values():
        sp.set_color('#ccc'); sp.set_linewidth(0.6)

    for ri, region in enumerate(regions_order_pre):
        df_rc = region_corr_pre[region]
        r_vals, p_vals = [], []
        for tl in transport_6labels:
            row = df_rc[(df_rc['transport'] == tl) & (df_rc['industry'] == industry)]
            r_vals.append(row['r'].values[0] if len(row) else np.nan)
            p_vals.append(row['p'].values[0] if len(row) else np.nan)

        xj     = x_pos + x_jitter[ri]
        color  = region_palette[region]
        marker = region_markers[region]

        for xi, (rv, pv) in enumerate(zip(r_vals, p_vals)):
            if np.isnan(rv): continue
            sig = not np.isnan(pv) and pv < 0.05
            ax.scatter(xj[xi], rv, s=110, color=color, marker=marker,
                       edgecolors='#222' if sig else color,
                       linewidths=1.8 if sig else 0.8,
                       alpha=0.95 if sig else 0.65, zorder=5)

    ax.axhline(0, color='#555', lw=1.2, ls='--', zorder=2)
    for xv in [0.5, 1.5, 2.5, 3.5, 4.5]:
        ax.axvline(xv, color='black', lw=2.0, ls='--', zorder=3)

    ax.set_xlim(-0.6, 5.6)
    ax.set_ylim(-1.15, 1.15)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(roman_labels, rotation=0, ha='center',
                       fontsize=FSIZE, fontfamily='Arial')
    ax.set_ylabel('Pearson r', fontsize=FSIZE, fontfamily='Arial')
    ax.set_yticks([-1, -0.5, 0, 0.5, 1])
    ax.tick_params(labelsize=FSIZE)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontfamily('Arial')
    ax.set_title(ghi_title[col_idx], fontsize=FSIZE + 3, fontweight='bold',
                 fontfamily='Arial', pad=8)
    ax.text(-0.17, 1.04, chr(ord('g') + col_idx), transform=ax.transAxes,
            fontsize=35, fontweight='bold', fontfamily='Arial', va='top')


lh_row1 = [
    mpatches.Patch(fc=sector_palette['Agriculture'], ec='white', lw=0.5,
                   alpha=0.95, label='AVA'),
    mpatches.Patch(fc=sector_palette['Industry'], ec='white', lw=0.5,
                   alpha=0.95, label='IVA'),
    mpatches.Patch(fc=sector_palette['Services'], ec='white', lw=0.5,
                   alpha=0.95, label='SVA'),
    mlines.Line2D([], [], color='#222', lw=1.8, label='Zero axis'),
    mpatches.Patch(fc='#888', ec='white', lw=0.5, alpha=0.75,
                   hatch='///', label='Negative r'),
    mlines.Line2D([], [], color='none', marker='$*$', ms=12,
                  markerfacecolor='#111', markeredgecolor='#111',
                  lw=0, label='p < 0.05 (map)'),
    mpatches.Patch(fc='white', ec='white', lw=0, label=''),
]

regions_order = sorted(region_palette.keys())
lh_row2 = [
    mlines.Line2D([], [], color=region_palette[r],
                  marker=region_markers[r], ms=10, lw=0,
                  label=r, markeredgecolor=region_palette[r], markeredgewidth=0.8)
    for r in regions_order
]
lh_row2 += [
    mlines.Line2D([], [], color='#555', marker='o', ms=10, lw=0,
                  markeredgecolor='#222', markeredgewidth=1.8,
                  label='p < 0.05'),
    mlines.Line2D([], [], color='#aaa', marker='o', ms=10, lw=0,
                  markeredgecolor='#aaa', markeredgewidth=0.8,
                  label='p ≥ 0.05'),
]

lh_row3 = [
    mpatches.Patch(fc='white', ec='white', lw=0, label=f'{rm}: {tl}')
    for rm, tl in zip(roman_labels, transport_6labels)
]
lh_row3.append(mpatches.Patch(fc='white', ec='white', lw=0, label=''))

all_handles = lh_row1 + lh_row2 + lh_row3
all_labels = [h.get_label() for h in all_handles]

ncol = 7

handles_row_first = [h for i in range(ncol) for h in all_handles[i::ncol]]
labels_row_first  = [l for i in range(ncol) for l in all_labels[i::ncol]]
# ────────────────────────

LSIZE = 21

leg = fig.legend(handles=handles_row_first,
                 labels=labels_row_first,
                 loc='lower center',
                 bbox_to_anchor=(0.5, 0.003),
                 bbox_transform=fig.transFigure,
                 ncol=ncol, fontsize=LSIZE, frameon=True, framealpha=.95,
                 edgecolor='#ccc', columnspacing=0.8,
                 handlelength=1.4, handletextpad=0.4,
                 prop={'family': 'Arial', 'size': LSIZE})

shift_pt = -(1.4 + 0.4) * LSIZE + 8
shift_trans = mtransforms.ScaledTranslation(shift_pt / 72, 0, fig.dpi_scale_trans)

for txt in leg.get_texts():
    label_text = txt.get_text()
    if any(label_text.startswith(f'{rm}:') for rm in roman_labels):
        txt.set_transform(txt.get_transform() + shift_trans)

plt.show()


In [ ]:
# Cell | Growth-rate helper functions

from scipy.stats import norm as sp_norm

# Calculate percentage growth relative to the selected baseline year.
def calc_growth_rate(val, base_val):
    """Calculate percentage growth relative to a baseline value."""
    if pd.isna(base_val) or pd.isna(val):
        return np.nan
    if base_val == 0:
        return val - base_val
    return (val - base_val) / abs(base_val) * 100

def fisher_stouffer(r_list, p_list):
    z_fisher = [0.5 * np.log((1+r)/(1-r)) for r in r_list if abs(r) < 1]
    if not z_fisher:
        return np.nan, np.nan
    z_mean = float(np.mean(z_fisher))
    r_mean = float((np.exp(2*z_mean) - 1) / (np.exp(2*z_mean) + 1))
    z_scores = [sp_norm.ppf(1 - min(pv, 0.9999)) for pv in p_list]
    p_combined = 1 - sp_norm.cdf(float(np.sum(z_scores) / np.sqrt(len(z_scores))))
    return r_mean, p_combined

transport_gr_cols = [f'{c}_gr' for c in transport_6cols]
industry_gr_cols  = [f'{c}_gr' for c in industry_cols3]

def compute_fisher_r(df_gr, entity_col):
    records = []
    entities = df_gr[entity_col].unique() if entity_col else [None]
    for tc_gr, tl in zip(transport_gr_cols, transport_6labels):
        for ic_gr, il in zip(industry_gr_cols, industry_labels3):
            r_list, p_list = [], []
            for ent in entities:
                df_e = df_gr[(df_gr[entity_col] == ent) & (df_gr['year'] >= 2016)] if entity_col else df_gr[df_gr['year'] >= 2016]
                sub = df_e[[tc_gr, ic_gr]].dropna()
                if len(sub) >= 4 and sub[tc_gr].std() > 0 and sub[ic_gr].std() > 0:
                    r_val, p_val = pearsonr(sub[tc_gr], sub[ic_gr])
                    r_list.append(r_val); p_list.append(p_val)
            r_agg, p_agg = fisher_stouffer(r_list, p_list)
            records.append({'transport': tl, 'industry': il,
                            'r': r_agg, 'p': p_agg, 'n': len(r_list)})
    return pd.DataFrame(records)

df_corr_A_country = compute_fisher_r(df_growth_country, 'country')

df_regional_lag = df_merged.groupby(['region', 'year']).agg(
    {**{c: 'mean' for c in transport_6cols},
     **{c: 'mean' for c in industry_cols3}}
).reset_index()

region_gr_records = []
for region in df_regional_lag['region'].unique():
    df_r = df_regional_lag[df_regional_lag['region'] == region].sort_values('year')
    base = df_r[df_r['year'] == 2015]
    if base.empty: continue
    base_row = base.iloc[0]
    for _, row in df_r.iterrows():
        rec = {'region': region, 'year': row['year']}
        for col in transport_6cols + industry_cols3:
            rec[f'{col}_gr'] = calc_growth_rate(row[col], base_row[col])
        region_gr_records.append(rec)
df_growth_region = pd.DataFrame(region_gr_records)

col_order = ['Agriculture', 'Industry', 'Services']
regions_order_5 = sorted(df_merged['region'].unique())

def make_pivots(df_corr):
    piv_r  = df_corr.pivot(index='transport', columns='industry', values='r').reindex(transport_6labels)[col_order]
    piv_p  = df_corr.pivot(index='transport', columns='industry', values='p').reindex(transport_6labels)[col_order]
    piv_sd = piv_r.std(axis=1)
    return piv_r, piv_p, piv_sd

FTICK, FTITLE = 20, 23
CMAP_RD = 'RdBu_r'

def draw_heatmap(ax, piv_r, piv_p, label, title):
    mat = piv_r.values.astype(float)
    vabs = max(np.nanmax(np.abs(mat)), 0.01)
    im = ax.imshow(mat, cmap=CMAP_RD, vmin=-vabs, vmax=vabs, aspect='auto')
    ax.set_xticks(range(3)); ax.set_xticklabels(col_order, fontsize=FTICK)
    ax.set_yticks(range(6)); ax.set_yticklabels(transport_6labels, fontsize=FTICK)
    for i in range(6):
        for j in range(3):
            val = mat[i, j]
            if np.isnan(val): continue
            p = piv_p.values[i, j]
            tc = 'white' if abs(val) > 0.45 else '#222'
            ax.text(j, i, f'{val:.2f}{"*" if p < 0.05 else ""}',
                    ha='center', va='center', fontsize=18,
                    fontfamily='Times New Roman', color=tc, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).ax.tick_params(labelsize=16)
    ax.set_title(title, fontsize=FTITLE, fontfamily='Times New Roman', pad=8)
    ax.text(-0.18, 1.01, label, transform=ax.transAxes,
            fontsize=28, fontweight='bold', fontfamily='Times New Roman', va='top')
    for y_sep in [2.5, 4.5]:
        ax.axhline(y_sep, color='#333', lw=2.0, ls='--')
    ax.text(-0.52, 1.25, 'Road &\nRailway', transform=ax.transAxes,
            fontsize=16, fontfamily='Times New Roman', color='#1A5276',
            ha='center', va='center', rotation=90)
    ax.text(-0.52, 0.25, 'Port &\nAirport', transform=ax.transAxes,
            fontsize=16, fontfamily='Times New Roman', color='#922B21',
            ha='center', va='center', rotation=90)

def draw_specificity(ax, piv_r, piv_sd, label, title):
    bar_x = np.arange(len(transport_6labels))
    bar_h = piv_sd.values
    ax.bar(bar_x, bar_h, width=0.55,
           color=['#2980B9' if sd < 0.15 else '#E07B39' for sd in bar_h],
           edgecolor='white', lw=0.8, alpha=0.85, zorder=3)
    for j, il in enumerate(col_order):
        ax.scatter(bar_x + (j-1)*0.14, piv_r[il].values, s=90,
                   color=sector_palette[il], marker=sector_markers[il],
                   edgecolors='white', lw=0.8, label=il, zorder=5)
    ax.axhline(0.15, color='#555', lw=1.4, ls=':', zorder=4)
    ax.text(5.55, 0.155, 'Broad\nthreshold', ha='right', va='bottom',
            fontsize=14, fontfamily='Times New Roman', color='#555')
    ax.set_xticks(bar_x)
    ax.set_xticklabels(transport_6labels, rotation=30, ha='right', fontsize=FTICK - 2)
    ax.set_ylabel('SD of r across sectors', fontsize=FTITLE - 2, fontfamily='Times New Roman')
    ax.set_title(title, fontsize=FTITLE, fontfamily='Times New Roman', pad=8)
    ax.tick_params(labelsize=FTICK - 2)
    ax.set_xlim(-0.6, 5.6); ax.set_ylim(bottom=0)
    ax.set_facecolor('#FAFAFA')
    for sp in ['top', 'right']: ax.spines[sp].set_visible(False)
    ax.text(-0.18, 1.01, label, transform=ax.transAxes,
            fontsize=28, fontweight='bold', fontfamily='Times New Roman', va='top')
    ax.axvline(2.5, color='#333', lw=1.6, ls='--', zorder=2)
    ax.axvline(3.5, color='#333', lw=1.6, ls='--', zorder=2)
    ax.annotate('', xy=(2.5, -0.065), xytext=(-0.5, -0.065),
                xycoords=('data', 'axes fraction'), textcoords=('data', 'axes fraction'),
                arrowprops=dict(arrowstyle='-', color='#1A5276', lw=2))
    ax.text(1.0, -0.10, 'Road & Railway', ha='center', va='top',
            transform=ax.get_xaxis_transform(),
            fontsize=16, fontfamily='Times New Roman', color='#1A5276')
    ax.annotate('', xy=(5.5, -0.065), xytext=(3.5, -0.065),
                xycoords=('data', 'axes fraction'), textcoords=('data', 'axes fraction'),
                arrowprops=dict(arrowstyle='-', color='#922B21', lw=2))
    ax.text(4.5, -0.10, 'Port & Airport', ha='center', va='top',
            transform=ax.get_xaxis_transform(),
            fontsize=16, fontfamily='Times New Roman', color='#922B21')

piv_r_c, piv_p_c, piv_sd_c = make_pivots(df_corr_A_country)

fig_Ac, axes_Ac = plt.subplots(1, 3, figsize=(24, 6),
    gridspec_kw={'wspace': 0.38, 'left': 0.06, 'right': 0.97, 'top': 0.88, 'bottom': 0.22})
fig_Ac.patch.set_facecolor('white')

draw_heatmap(axes_Ac[0], piv_r_c, piv_p_c, 'g',
             'Fisher Z mean r\n(country growth rate time-series)')

ax = axes_Ac[1]
highlight = [('total_road_km_gr', 'Road', '#1A5276'), ('port_area_km2_gr', 'Port', '#922B21')]
box_data, box_labels, box_colors = [], [], []
for tc_gr, tl, col in highlight:
    r_vals = []
    for country in df_growth_country['country'].unique():
        df_c = df_growth_country[df_growth_country['country'] == country]
        sub = df_c[[tc_gr, 'Services_value_added_gr']].dropna()
        if len(sub) >= 4 and sub[tc_gr].std() > 0 and sub['Services_value_added_gr'].std() > 0:
            r_vals.append(pearsonr(sub[tc_gr], sub['Services_value_added_gr'])[0])
    box_data.append(r_vals); box_labels.append(tl); box_colors.append(col)
bp = ax.boxplot(box_data, patch_artist=True, widths=0.45,
                medianprops=dict(color='white', lw=2.5))
for patch, col in zip(bp['boxes'], box_colors):
    patch.set_facecolor(col); patch.set_alpha(0.75)
for whisker in bp['whiskers']: whisker.set(color='#555', lw=1.2)
for cap in bp['caps']: cap.set(color='#555', lw=1.2)
for flier in bp['fliers']: flier.set(marker='o', color='#888', alpha=0.5, ms=4)
for xi, (r_vals, col) in enumerate(zip(box_data, box_colors), start=1):
    jitter = np.random.default_rng(42).uniform(-0.12, 0.12, len(r_vals))
    ax.scatter(xi + jitter, r_vals, s=30, color=col, alpha=0.55,
               edgecolors='white', lw=0.5, zorder=4)
ax.axhline(0, color='#555', lw=1.2, ls='--')
ax.set_xticks([1, 2]); ax.set_xticklabels(box_labels, fontsize=FTICK)
ax.set_ylabel('Pearson r (per country)', fontsize=FTITLE - 2, fontfamily='Times New Roman')
ax.set_title('Road vs Port → Services\n(per-country r distribution)',
             fontsize=FTITLE, fontfamily='Times New Roman', pad=8)
ax.tick_params(labelsize=FTICK - 2); ax.set_facecolor('#FAFAFA')
for sp in ['top', 'right']: ax.spines[sp].set_visible(False)
ax.text(-0.18, 1.01, 'h', transform=ax.transAxes,
        fontsize=28, fontweight='bold', fontfamily='Times New Roman', va='top')

draw_specificity(axes_Ac[2], piv_r_c, piv_sd_c, 'i',
                 'Sector-specificity index\n(country scale)')

os.makedirs('outputs', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
plt.savefig(r'outputs/fig_transport_sector_specificity.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Initialize the figure layout.
fig_Ar, axes_Ar = plt.subplots(5, 3, figsize=(24, 30),
    gridspec_kw={'wspace': 0.38, 'hspace': 0.65,
                 'left': 0.06, 'right': 0.97, 'top': 0.97, 'bottom': 0.04})
fig_Ar.patch.set_facecolor('white')

label_idx = ord('j')
for ri, region in enumerate(regions_order_5):
    df_gr_r = df_growth_region[df_growth_region['region'] == region]

    df_countries_in_region = df_growth_country[
        df_growth_country['country'].map(country_to_region) == region
    ]
    df_corr_region_i = compute_fisher_r(df_countries_in_region, 'country')
    piv_r_ri, piv_p_ri, piv_sd_ri = make_pivots(df_corr_region_i)

    draw_heatmap(axes_Ar[ri, 0], piv_r_ri, piv_p_ri, chr(label_idx),
                 f'Fisher Z mean r — {region}\n(country growth rate time-series)')
    label_idx += 1

    ax = axes_Ar[ri, 1]
    box_data_ri, box_labels_ri, box_colors_ri = [], [], []
    for tc_gr, tl, col in highlight:
        r_vals = []
        for country in df_countries_in_region['country'].unique():
            df_c = df_countries_in_region[df_countries_in_region['country'] == country]
            sub = df_c[[tc_gr, 'Services_value_added_gr']].dropna()
            if len(sub) >= 4 and sub[tc_gr].std() > 0 and sub['Services_value_added_gr'].std() > 0:
                r_vals.append(pearsonr(sub[tc_gr], sub['Services_value_added_gr'])[0])
        box_data_ri.append(r_vals); box_labels_ri.append(tl); box_colors_ri.append(col)
    bp_ri = ax.boxplot(box_data_ri, patch_artist=True, widths=0.45,
                       medianprops=dict(color='white', lw=2.5))
    for patch, col in zip(bp_ri['boxes'], box_colors_ri):
        patch.set_facecolor(col); patch.set_alpha(0.75)
    for whisker in bp_ri['whiskers']: whisker.set(color='#555', lw=1.2)
    for cap in bp_ri['caps']: cap.set(color='#555', lw=1.2)
    for flier in bp_ri['fliers']: flier.set(marker='o', color='#888', alpha=0.5, ms=4)
    for xi, (r_vals, col) in enumerate(zip(box_data_ri, box_colors_ri), start=1):
        if r_vals:
            jitter = np.random.default_rng(42).uniform(-0.12, 0.12, len(r_vals))
            ax.scatter(xi + jitter, r_vals, s=50, color=col, alpha=0.7,
                       edgecolors='white', lw=0.5, zorder=4)
    ax.axhline(0, color='#555', lw=1.2, ls='--')
    ax.set_xticks([1, 2]); ax.set_xticklabels(box_labels_ri, fontsize=FTICK)
    ax.set_ylabel('Pearson r (per country)', fontsize=FTITLE - 2, fontfamily='Times New Roman')
    ax.set_title(f'Road vs Port → Services — {region}\n(per-country r distribution)',
                 fontsize=FTITLE, fontfamily='Times New Roman', pad=8)
    ax.tick_params(labelsize=FTICK - 2); ax.set_facecolor('#FAFAFA')
    for sp in ['top', 'right']: ax.spines[sp].set_visible(False)
    ax.text(-0.18, 1.01, chr(label_idx), transform=ax.transAxes,
            fontsize=28, fontweight='bold', fontfamily='Times New Roman', va='top')
    label_idx += 1

    draw_specificity(axes_Ar[ri, 2], piv_r_ri, piv_sd_ri, chr(label_idx),
                     f'Sector-specificity — {region}')
    label_idx += 1

plt.show()


In [ ]:
# Cell | Continental sectoral-growth aggregation

agg_transport = df_merged.groupby('year')[transport_6cols].sum()
agg_industry  = df_merged.groupby('year')[industry_cols3].mean()
df_continent  = pd.concat([agg_transport, agg_industry], axis=1).reset_index()

base_cont = df_continent[df_continent['year'] == 2015].iloc[0]
cont_gr_records = []
for _, row in df_continent.iterrows():
    rec = {'year': row['year']}
    for col in transport_6cols + industry_cols3:
        rec[f'{col}_gr'] = calc_growth_rate(row[col], base_cont[col])
    cont_gr_records.append(rec)
df_growth_continent = pd.DataFrame(cont_gr_records)

df_regional_ts = df_merged.groupby(['region', 'year']).agg(
    {**{c: 'sum' for c in transport_6cols},
     **{c: 'mean' for c in industry_cols3}}
).reset_index()

reg_ts_gr_records = []
for region in df_regional_ts['region'].unique():
    df_r = df_regional_ts[df_regional_ts['region'] == region].sort_values('year')
    base = df_r[df_r['year'] == 2015]
    if base.empty: continue
    base_row = base.iloc[0]
    for _, row in df_r.iterrows():
        rec = {'region': region, 'year': row['year']}
        for col in transport_6cols + industry_cols3:
            rec[f'{col}_gr'] = calc_growth_rate(row[col], base_row[col])
        reg_ts_gr_records.append(rec)
df_growth_regional_ts = pd.DataFrame(reg_ts_gr_records)

cont_records = []
for tc_gr, tl in zip(transport_gr_cols, transport_6labels):
    for ic_gr, il in zip(industry_gr_cols, industry_labels3):
        sub = df_growth_continent[df_growth_continent["year"] >= 2016][[tc_gr, ic_gr]].dropna()
        if len(sub) >= 4 and sub[tc_gr].std() > 0 and sub[ic_gr].std() > 0:
            r_val, p_val = pearsonr(sub[tc_gr], sub[ic_gr])
        else:
            r_val, p_val = np.nan, np.nan
        cont_records.append({'transport': tl, 'industry': il, 'r': r_val, 'p': p_val, 'n': len(sub)})
df_corr_C_cont = pd.DataFrame(cont_records)

regions_order_5 = sorted(df_growth_regional_ts['region'].unique())
region_corr_C = {}
for region in regions_order_5:
    df_r = df_growth_regional_ts[(df_growth_regional_ts['region'] == region) & (df_growth_regional_ts['year'] >= 2016)]
    records = []
    for tc_gr, tl in zip(transport_gr_cols, transport_6labels):
        for ic_gr, il in zip(industry_gr_cols, industry_labels3):
            sub = df_r[[tc_gr, ic_gr]].dropna()
            if len(sub) >= 4 and sub[tc_gr].std() > 0 and sub[ic_gr].std() > 0:
                r_val, p_val = pearsonr(sub[tc_gr], sub[ic_gr])
            else:
                r_val, p_val = np.nan, np.nan
            records.append({'transport': tl, 'industry': il, 'r': r_val, 'p': p_val})
    region_corr_C[region] = pd.DataFrame(records)

col_order = ['Agriculture', 'Industry', 'Services']
piv_r_cont, piv_p_cont, piv_sd_cont = make_pivots(df_corr_C_cont)

print(piv_r_cont.round(3).to_string())

FTICK, FTITLE = 20, 23

fig_Cc, axes_Cc = plt.subplots(1, 3, figsize=(24, 6),
    gridspec_kw={'wspace': 0.38, 'left': 0.06, 'right': 0.97, 'top': 0.88, 'bottom': 0.22})
fig_Cc.patch.set_facecolor('white')

draw_heatmap(axes_Cc[0], piv_r_cont, piv_p_cont, 'g',
             'Pearson r (continent growth rate time-series)')

ax = axes_Cc[1]
years = df_growth_continent['year'].values
ax2_twin = ax.twinx()
for tc_gr, tl, col in [('total_road_km_gr', 'Road', '#1A5276'), ('port_area_km2_gr', 'Port', '#922B21')]:
    ax.plot(years, df_growth_continent[tc_gr].values, color=col, lw=2.2, marker='o', ms=5, label=tl, zorder=4)
ax2_twin.plot(years, df_growth_continent['Services_value_added_gr'].values,
              color='#C0392B', lw=2.2, ls='--', marker='^', ms=5, label='Services', zorder=3)
ax.set_xlabel('Year', fontsize=FTITLE - 2, fontfamily='Times New Roman')
ax.set_ylabel('Transport growth rate (%)', fontsize=FTITLE - 2, fontfamily='Times New Roman')
ax2_twin.set_ylabel('Services growth rate (%)', fontsize=FTITLE - 2, fontfamily='Times New Roman',
                    color='#C0392B')
ax2_twin.tick_params(labelsize=FTICK - 2, colors='#C0392B')
ax.set_title('Road & Port vs Services\n(continent growth rate time-series)',
             fontsize=FTITLE, fontfamily='Times New Roman', pad=8)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=15,
          prop={'family': 'Times New Roman', 'size': 15}, frameon=True, framealpha=0.9)
ax.tick_params(labelsize=FTICK - 2); ax.set_facecolor('#FAFAFA')
for sp in ['top']: ax.spines[sp].set_visible(False)
ax.text(-0.18, 1.01, 'h', transform=ax.transAxes,
        fontsize=28, fontweight='bold', fontfamily='Times New Roman', va='top')

draw_specificity(axes_Cc[2], piv_r_cont, piv_sd_cont, 'i',
                 'Sector-specificity index\n(continent growth rate)')

plt.show()

# Initialize the figure layout.
fig_Cr, axes_Cr = plt.subplots(5, 3, figsize=(24, 30),
    gridspec_kw={'wspace': 0.38, 'hspace': 0.65,
                 'left': 0.06, 'right': 0.97, 'top': 0.97, 'bottom': 0.04})
fig_Cr.patch.set_facecolor('white')

label_idx = ord('j')
for ri, region in enumerate(regions_order_5):
    df_corr_ri = region_corr_C[region]
    piv_r_ri, piv_p_ri, piv_sd_ri = make_pivots(df_corr_ri)

    draw_heatmap(axes_Cr[ri, 0], piv_r_ri, piv_p_ri, chr(label_idx),
                 f'Pearson r — {region}\n(regional aggregated growth rate time-series)')
    label_idx += 1

    ax = axes_Cr[ri, 1]
    df_r = df_growth_regional_ts[df_growth_regional_ts['region'] == region].sort_values('year')
    years_r = df_r['year'].values
    ax2_twin = ax.twinx()
    for tc_gr, tl, col in [('total_road_km_gr', 'Road', '#1A5276'), ('port_area_km2_gr', 'Port', '#922B21')]:
        ax.plot(years_r, df_r[tc_gr].values, color=col, lw=2.2, marker='o', ms=5, label=tl, zorder=4)
    ax2_twin.plot(years_r, df_r['Services_value_added_gr'].values,
                  color='#C0392B', lw=2.2, ls='--', marker='^', ms=5, label='Services', zorder=3)
    ax.set_xlabel('Year', fontsize=FTITLE - 2, fontfamily='Times New Roman')
    ax.set_ylabel('Transport growth rate (%)', fontsize=FTITLE - 2, fontfamily='Times New Roman')
    ax2_twin.set_ylabel('Services growth rate (%)', fontsize=FTITLE - 2, fontfamily='Times New Roman',
                        color='#C0392B')
    ax2_twin.tick_params(labelsize=FTICK - 2, colors='#C0392B')
    ax.set_title(f'Road & Port vs Services — {region}\n(regional growth rate time-series)',
                 fontsize=FTITLE, fontfamily='Times New Roman', pad=8)
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2_twin.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=13,
              prop={'family': 'Times New Roman', 'size': 13}, frameon=True, framealpha=0.9)
    ax.tick_params(labelsize=FTICK - 2); ax.set_facecolor('#FAFAFA')
    for sp in ['top']: ax.spines[sp].set_visible(False)
    ax.text(-0.18, 1.01, chr(label_idx), transform=ax.transAxes,
            fontsize=28, fontweight='bold', fontfamily='Times New Roman', va='top')
    label_idx += 1

    draw_specificity(axes_Cr[ri, 2], piv_r_ri, piv_sd_ri, chr(label_idx),
                     f'Sector-specificity — {region}')
    label_idx += 1

plt.show()
